# 🔬 5-Way LLM Comparison: RAG | KG | Agent | Finetuned | Hybrid (All-on-Finetuned)
Each approach answers the same questions. Approach 5 is a hybrid that chains RAG + KG + Agent reasoning on top of your finetuned model. Final cell shows an interactive dashboard.

## Cell 1 — Install Dependencies

In [1]:
!pip install faiss-cpu sentence-transformers transformers torch pypdf networkx

## Cell 2 — Shared Utilities (used by all 4 approaches)

In [2]:
import os, re, json, time
import numpy as np
import torch
import faiss
import networkx as nx
from collections import Counter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

INPUT_DIR = "./documents"
MODEL_ID  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_PATH   = "out/tinyllama-finetuned/final"

# ── Text extraction ──────────────────────────────────────────────────────────
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    return "".join(page.extract_text() or "" for page in reader.pages)

def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += chunk_size - overlap
        if len(words) - start < 50:
            break
    return chunks

def is_bibliography_chunk(chunk):
    """
    Returns True if a chunk is mostly citations/references.
    FIX for original RAG issue: bibliography chunks were being retrieved
    instead of actual content, causing irrelevant answers.
    Detection heuristics:
      - Too many bracket pairs  → reference list e.g. [1], [2]
      - Too many DOI/URL links  → bibliography entries
      - High ratio of year patterns → dense citation text
    """
    bracket_count = chunk.count("[") + chunk.count("]")
    doi_count     = chunk.lower().count("doi") + chunk.lower().count("https://")
    year_matches  = re.findall(r'\b(19|20)\d{2}\b', chunk)
    word_count    = max(len(chunk.split()), 1)
    return (bracket_count > 10) or (doi_count > 3) or (len(year_matches) / word_count > 0.08)

def load_all_documents(input_dir=INPUT_DIR):
    """Load PDFs, chunk them, filter bibliography chunks."""
    all_chunks, metadata, raw_texts = [], [], {}
    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(".pdf"):
            continue
        path = os.path.join(input_dir, filename)
        print(f"Loading: {filename}")
        text = extract_text_from_pdf(path)
        raw_texts[filename] = text
        chunks = chunk_text(text)
        kept = 0
        for idx, chunk in enumerate(chunks):
            if is_bibliography_chunk(chunk):
                continue
            all_chunks.append(chunk)
            metadata.append({"filename": filename, "chunk_index": idx})
            kept += 1
        print(f"  {len(chunks)} chunks → {kept} kept (bibliography filtered)")
    print(f"\nTotal content chunks: {len(all_chunks)}")
    return all_chunks, metadata, raw_texts

# ── Scoring helpers ───────────────────────────────────────────────────────────
def relevance_score(answer, question):
    """Keyword overlap score. Higher = answer is more topically on-point."""
    stopwords = {"what","is","the","a","an","how","why","does","do",
                 "in","of","and","to","was","for","are","this","that"}
    q_words = {w.lower().strip("?,.") for w in question.split() if w.lower() not in stopwords}
    a_words = {w.lower().strip("?,.") for w in answer.split()}
    return round(len(q_words & a_words) / max(len(q_words), 1), 3)

def has_repetition_loop(text, threshold=3):
    """Detect if the model fell into a repetition loop (e.g. ### Response: x20)."""
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return any(v >= threshold for v in Counter(lines).values()) if lines else False

print("✅ Shared utilities ready")

✅ Shared utilities ready


In [3]:
# ── Kill any leftover GPU processes from previous runs ───────────────────────
import gc
import torch

# Clear Python garbage
gc.collect()
torch.cuda.empty_cache()

# Show what's using GPU memory before we start
print("GPU memory before cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  Reserved  : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"  Free      : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")

GPU memory before cleanup:
  Allocated : 0.00 GB
  Reserved  : 0.00 GB
  Free      : 15.64 GB


## Cell 3 — Load Models and Documents

## Cell 3 — Load Models and Documents (OOM-safe)

In [4]:
import gc
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def gpu_stats(label=""):
    alloc  = torch.cuda.memory_allocated() / 1e9
    reserv = torch.cuda.memory_reserved() / 1e9
    free   = torch.cuda.get_device_properties(0).total_memory / 1e9 - reserv
    print(f"  GPU [{label}] allocated:{alloc:.2f}GB reserved:{reserv:.2f}GB free:{free:.2f}GB")

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

# ── Step 1: Documents (CPU only) ──────────────────────────────────────────────
all_chunks, chunk_metadata, raw_texts = load_all_documents()

# ── Step 2: Embedder on CPU — saves ~300MB GPU for the LLMs ──────────────────
print("\nLoading embedder on CPU...")
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("✅ Embedder ready (CPU)")
gpu_stats("after embedder")

# ── Step 3: Load finetuned FIRST — litgpt always loads to GPU ─────────────────
# Load it first while GPU is empty, then immediately move to CPU.
print("\nLoading finetuned model (needs empty GPU)...")
free_gpu()
gpu_stats("before finetuned")
from litgpt import LLM
finetuned_llm = LLM.load(FT_PATH)
print("✅ Finetuned model loaded")
gpu_stats("after finetuned")

# ── Step 4: Move finetuned to CPU to free GPU for base model ──────────────────
print("\nMoving finetuned to CPU to make room...")
if hasattr(finetuned_llm, "model"):
    finetuned_llm.model = finetuned_llm.model.to("cpu")
free_gpu()
gpu_stats("after moving finetuned to CPU")

# ── Step 5: Load base TinyLlama onto GPU ──────────────────────────────────────
print("\nLoading base TinyLlama onto GPU...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
base_model.eval()
print("✅ Base TinyLlama ready")
gpu_stats("after base model")

# ── Step 6: Model switcher ────────────────────────────────────────────────────
# Call use_base()      before RAG / KG / Agent calls
# Call use_finetuned() before Finetuned / Hybrid calls
_active = "base"

def use_base():
    global _active
    if _active == "base":
        return
    if hasattr(finetuned_llm, "model"):
        finetuned_llm.model = finetuned_llm.model.to("cpu")
    free_gpu()
    base_model.to("cuda")
    _active = "base"
    print("  ↔ Switched to base model")

def use_finetuned():
    global _active
    if _active == "finetuned":
        return
    base_model.to("cpu")
    free_gpu()
    if hasattr(finetuned_llm, "model"):
        finetuned_llm.model = finetuned_llm.model.to("cuda")
    _active = "finetuned"
    print("  ↔ Switched to finetuned model")

# ── Step 7: Shared generate function ─────────────────────────────────────────
def generate(prompt, max_new_tokens=300):
    """Generate with base TinyLlama. Always call use_base() before this."""
    chat = (
        "<|system|>You are a helpful assistant that answers questions "
        f"based on provided context.</s>\n<|user|>{prompt}</s>\n<|assistant|>"
    )
    device = next(base_model.parameters()).device
    inputs = tokenizer(
        chat, return_tensors="pt", truncation=True, max_length=1500
    ).to(device)
    with torch.no_grad():
        out = base_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("\n✅ All models ready. Switcher active.")
print("   use_base()      → RAG / KG / Agent calls")
print("   use_finetuned() → Finetuned / Hybrid calls")

Loading: 3545008.3545087.pdf
  25 chunks → 6 kept (bibliography filtered)
Loading: applsci-12-02160-v2.pdf
  19 chunks → 15 kept (bibliography filtered)
Loading: 3458817.3476223.pdf
  42 chunks → 28 kept (bibliography filtered)
Loading: Paper1.pdf
  12 chunks → 10 kept (bibliography filtered)
Loading: DBA_Residency_hsampatirao.pdf
  2 chunks → 1 kept (bibliography filtered)
Loading: Paper2.pdf
  11 chunks → 9 kept (bibliography filtered)
Loading: Hariprasad_Sampatirao_Draft_Chapter1.pdf
  25 chunks → 23 kept (bibliography filtered)
Loading: atc23-weng.pdf
  33 chunks → 27 kept (bibliography filtered)
Loading: 3638757.pdf
  54 chunks → 5 kept (bibliography filtered)
Loading: Walsh_Dissertation_Data Readiness.pdf
  15 chunks → 12 kept (bibliography filtered)

Total content chunks: 136

Loading embedder on CPU...
✅ Embedder ready (CPU)
  GPU [after embedder] allocated:0.00GB reserved:0.00GB free:15.64GB

Loading finetuned model (needs empty GPU)...
  GPU [before finetuned] allocated:0.00G

`torch_dtype` is deprecated! Use `dtype` instead!


  GPU [after moving finetuned to CPU] allocated:0.00GB reserved:0.00GB free:15.64GB

Loading base TinyLlama onto GPU...
✅ Base TinyLlama ready
  GPU [after base model] allocated:2.20GB reserved:2.35GB free:13.29GB

✅ All models ready. Switcher active.
   use_base()      → RAG / KG / Agent calls
   use_finetuned() → Finetuned / Hybrid calls


## Cell 4 — Approach 1: Fixed RAG (FAISS + bibliography filter)

In [5]:
# Build FAISS index over filtered content chunks
print("Building FAISS index...")
embeddings = embedder.encode(all_chunks, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(embeddings)
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"✅ FAISS: {faiss_index.ntotal} vectors")

def rag_retrieve(query, top_k=3):
    """Embed query, find top_k nearest chunks in FAISS index."""
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    dists, idxs = faiss_index.search(q_vec, top_k)
    return [
        {"chunk": all_chunks[i], "source": chunk_metadata[i]["filename"], "score": float(d)}
        for d, i in zip(dists[0], idxs[0])
    ]

def rag_answer(question, top_k=3):
    """RAG pipeline: Retrieve relevant chunks → build context → generate answer."""
    use_base()  # ensure base model is on GPU
    results = rag_retrieve(question, top_k)
    context = "\n\n---\n\n".join(f"[{r['source']}]\n{r['chunk']}" for r in results)
    prompt  = f"Based on the following context, answer the question clearly.\n\nContext:\n{context}\n\nQuestion: {question}"
    return generate(prompt), results

# Quick test
test_ans, _ = rag_answer("What is data readiness?")
print(f"RAG test (150 chars): {test_ans[:150]}...")

Building FAISS index...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ FAISS: 136 vectors
RAG test (150 chars): m/alibaba/clusterda/tree/main/benchmark/production/v2025/v2025-instance_id_role_cpu_request_gpu_request_memory_request_rdma_request_disk_request_creat...


## Cell 5 — Approach 2: Knowledge Graph (NetworkX)

In [6]:
#
# How Knowledge Graph differs from RAG:
#
#   RAG asks: "Which chunks are SIMILAR to my query?" (vector distance)
#   KG asks:  "Which CONCEPTS match my query, and what are they connected to?"
#
# The graph captures relationships between ideas, not just text similarity.
# A chunk about 'data quality' and a chunk about 'model performance' get
# linked if they co-occur often — KG can traverse that link, RAG cannot.
#
# Graph structure:
#   Nodes = concepts (key terms extracted from chunks)
#   Edges = co-occurrence in same chunk (weight = frequency)

def extract_concepts(text, max_concepts=15):
    """Extract key concepts using capitalized phrase patterns + word frequency."""
    # Capitalized multi-word phrases (e.g. 'Data Readiness', 'Machine Learning')
    cap_phrases = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', text)
    # High-frequency content words
    stopwords = {"this","that","with","from","they","have","been","which",
                 "were","their","also","more","than","into","when","about"}
    words = [w.lower() for w in re.findall(r'\b[a-zA-Z]{4,}\b', text)]
    freq  = {}
    for w in words:
        if w not in stopwords:
            freq[w] = freq.get(w, 0) + 1
    top_words = [w for w, _ in sorted(freq.items(), key=lambda x: -x[1])[:10]]
    concepts  = list(dict.fromkeys([p.lower() for p in cap_phrases] + top_words))[:max_concepts]
    return [c for c in concepts if len(c) >= 4]

def build_knowledge_graph(chunks, metadata):
    """Build NetworkX graph: nodes=concepts, edges=co-occurrence in same chunk."""
    G = nx.DiGraph()
    concept_to_chunks = {}

    for chunk, meta in zip(chunks, metadata):
        concepts = extract_concepts(chunk)
        source   = meta["filename"]
        for concept in concepts:
            if concept not in G:
                G.add_node(concept, sources=set(), chunks=[])
            G.nodes[concept]["sources"].add(source)
            G.nodes[concept]["chunks"].append(chunk[:200])
            concept_to_chunks.setdefault(concept, []).append(chunk)
        # Connect all concepts that appear in the same chunk
        for i, c1 in enumerate(concepts):
            for c2 in concepts[i+1:]:
                if G.has_edge(c1, c2):
                    G[c1][c2]["weight"] += 1
                else:
                    G.add_edge(c1, c2, weight=1)

    print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G, concept_to_chunks

def kg_retrieve(question, G, concept_to_chunks, top_k=5):
    """Match question keywords to graph nodes, expand to neighbours, return chunks."""
    q_words = set(question.lower().split())
    # Step 1: Find nodes whose name contains a question keyword
    matched = [node for node in G.nodes if any(qw in node for qw in q_words)]
    # Step 2: Expand 1-hop to neighbours (sorted by edge weight)
    expanded = set(matched)
    for concept in matched:
        neighbours = list(G.successors(concept)) + list(G.predecessors(concept))
        weighted   = sorted(
            neighbours,
            key=lambda n: max(
                G[concept].get(n, {}).get("weight", 0),
                G.get_edge_data(n, concept, {}).get("weight", 0)
            ), reverse=True
        )
        expanded.update(weighted[:3])
    # Step 3: Collect unique chunks from expanded concept set
    results, seen = [], set()
    for concept in expanded:
        for chunk in concept_to_chunks.get(concept, []):
            if chunk not in seen:
                results.append({"chunk": chunk, "concept": concept})
                seen.add(chunk)
        if len(results) >= top_k:
            break
    return results, list(expanded)

def kg_answer(question):
    """KG pipeline: match concepts → expand graph → generate answer."""
    use_base()  # base model on GPU for generation
    retrieved, concepts_used = kg_retrieve(question, kg, concept_chunks)
    if not retrieved:
        fallback = rag_retrieve(question, top_k=2)
        context  = "\n\n---\n\n".join(r["chunk"] for r in fallback)
        concepts_used = ["(vector search fallback)"]
    else:
        context = "\n\n---\n\n".join(r["chunk"] for r in retrieved[:3])
    prompt = (
        f"Using knowledge graph context about related concepts, answer the question.\n\n"
        f"Related concepts: {', '.join(concepts_used[:8])}\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return generate(prompt), concepts_used

print("Building Knowledge Graph...")
kg, concept_chunks = build_knowledge_graph(all_chunks, chunk_metadata)
top10 = sorted(kg.degree(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 most connected concepts:")
for concept, degree in top10:
    print(f"  '{concept}' — {degree} connections")

Building Knowledge Graph...
Knowledge Graph: 884 nodes, 10141 edges

Top 10 most connected concepts:
  'cluster' — 286 connections
  'resource' — 265 connections
  'gpus' — 212 connections
  'scheduling' — 190 connections
  'data' — 183 connections
  'model' — 175 connections
  'time' — 164 connections
  'node' — 144 connections
  'tasks' — 144 connections
  'alibaba cluster trace' — 141 connections


## Cell 6 — Approach 3: Multi-Step Agent (Plan → Search → Reason → Answer)

In [7]:
#
# How the Multi-Step Agent differs from RAG and KG:
#
#   RAG:   1 retrieval call → 1 generation call  (fast, simple)
#   KG:    concept match → graph traversal → 1 generation call
#   Agent: 4 explicit reasoning steps, each informed by the previous
#
# Step 1 PLAN:   LLM decomposes the question into targeted sub-questions
# Step 2 SEARCH: Each sub-question gets its own FAISS retrieval independently
# Step 3 REASON: LLM reads all evidence and writes intermediate reasoning
# Step 4 ANSWER: LLM synthesises reasoning into a clean final answer
#
# This costs 4 LLM calls but handles complex multi-part questions better.

def agent_plan(question):
    """Step 1: Ask LLM to decompose question into 2-3 sub-questions."""
    prompt = (
        f"Break this research question into 2-3 specific sub-questions for document retrieval.\n"
        f"Return ONLY a numbered list, nothing else.\n\nQuestion: {question}\n\nSub-questions:"
    )
    raw = generate(prompt, max_new_tokens=150)
    sub_qs = []
    for line in raw.split("\n"):
        m = re.match(r'^[\d\-\*]+[\)\.]?\s+(.+)', line.strip())
        if m and len(m.group(1)) > 10:
            sub_qs.append(m.group(1).strip())
    return sub_qs[:3] if sub_qs else [question]  # fallback to original

def agent_search(sub_questions, top_k=2):
    """Step 2: Run independent FAISS retrieval for each sub-question."""
    results, seen = [], set()
    for sub_q in sub_questions:
        for r in rag_retrieve(sub_q, top_k=top_k):
            key = r["chunk"][:100]
            if key not in seen:
                r["sub_question"] = sub_q
                results.append(r)
                seen.add(key)
    return results

def agent_reason(question, sub_questions, evidence):
    """Step 3: LLM writes intermediate reasoning over all collected evidence."""
    evidence_text = "\n\n".join(
        f"[For: '{e['sub_question']}']\n{e['chunk'][:300]}" for e in evidence
    )
    prompt = (
        f"Reason through this question using the evidence. Write 3-4 sentences.\n\n"
        f"Question: {question}\nSub-questions: {'; '.join(sub_questions)}\n\n"
        f"Evidence:\n{evidence_text}\n\nReasoning:"
    )
    return generate(prompt, max_new_tokens=200)

def agent_synthesise(question, reasoning):
    """Step 4: Compose final answer from intermediate reasoning."""
    prompt = (
        f"Based on this reasoning, give a clear final answer.\n\n"
        f"Question: {question}\nReasoning: {reasoning}\n\nFinal answer:"
    )
    return generate(prompt, max_new_tokens=250)

def agent_answer(question):
    """Full 4-step agent pipeline. Returns (answer, trace)."""
    use_base()  # all 4 steps use base model
    trace = {}
    print("    Step 1: Planning...")
    sub_questions = agent_plan(question)
    trace["sub_questions"] = sub_questions
    print(f"      → {sub_questions}")
    print("    Step 2: Searching...")
    evidence = agent_search(sub_questions, top_k=2)
    trace["evidence_count"] = len(evidence)
    print(f"      → {len(evidence)} unique chunks")
    print("    Step 3: Reasoning...")
    reasoning = agent_reason(question, sub_questions, evidence)
    trace["reasoning"] = reasoning
    print("    Step 4: Synthesising...")
    answer = agent_synthesise(question, reasoning)
    trace["final_answer"] = answer
    return answer, trace

print("Testing agent pipeline...")
test_ans, test_trace = agent_answer("What is data readiness?")
print(f"\nAgent test (150 chars): {test_ans[:150]}...")

Testing agent pipeline...
    Step 1: Planning...
      → ['Definition: Data readiness is the state of being ready for data-driven decision-making.', 'Examples: In the case of a healthcare provider, data readiness can refer to the ability to access and analyze patient data for monitoring and treatment management.', 'Impact: Data readiness can affect business decision-making by enabling a quick and efficient analysis of data, allowing for more informed decisions.']
    Step 2: Searching...
      → 4 unique chunks
    Step 3: Reasoning...
    Step 4: Synthesising...

Agent test (150 chars): 
Based on the evidence presented in the text, data readiness refers to the state of being ready for data-driven decision-making, which is the ability ...


## Cell 7 — Approach 4: Finetuned TinyLlama

In [8]:
def finetuned_answer(question):
    """
    Finetuned TinyLlama with:
    1. RAG retrieval for context (same as other approaches)
    2. Alpaca prompt format (must match training format)
    3. Context truncation to fit 2048 token limit
    4. Post-processing to strip repetition loops
    """
    use_base()  # retrieval needs embedder (CPU) — base model must be on GPU for generate
    results = rag_retrieve(question, top_k=2)
    context = "\n\n---\n\n".join(r["chunk"] for r in results)
    words = context.split()
    if len(words) > 300:
        context = " ".join(words[:300]) + "..."
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\nBased on the following context, answer the question.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\n### Response:"
    )
    use_finetuned()  # swap to finetuned for generation
    answer = finetuned_llm.generate(prompt, max_new_tokens=300)
    if "### Response:" in answer:
        answer = answer.split("### Response:")[0].strip()
    return answer

test_ft = finetuned_answer("What is data readiness?")
print(f"Finetuned test (150 chars): {test_ft[:150]}...")

  ↔ Switched to finetuned model
Finetuned test (150 chars): Hariprasad Sampatirao
Zoloji University, USA

I can say that the text is helpful to understand the purpose and structure of the required documents for...


## Cell 8 — Approach 5: Hybrid (RAG + KG + Agent on Finetuned)

In [9]:
#
# Approach 5 — Hybrid: RAG + Knowledge Graph + Multi-Step Agent, all feeding
# into YOUR FINETUNED MODEL as the final answer generator.
#
# How it differs from the other 4 approaches:
#
#   Approaches 1-3: Base TinyLlama generates the answer
#   Approach 4:     Finetuned model generates, but with simple RAG context
#   Approach 5:     ALL retrieval strategies run first, their evidence is
#                   merged and deduplicated, then the finetuned model
#                   generates the final answer from the richer context.
#
# This tests the hypothesis:
#   "Does better context retrieval make the finetuned model answer better?"
#
# Pipeline:
#   Step 1 — RAG retrieval    → top-k semantically similar chunks
#   Step 2 — KG retrieval     → concept-linked chunks
#   Step 3 — Agent planning   → sub-questions → targeted retrieval
#   Step 4 — Merge & dedup    → combine all evidence, remove duplicates
#   Step 5 — Finetuned model  → generate final answer from merged context

def hybrid_retrieve(question, top_k=2):
    """
    Runs all 3 retrieval strategies and merges their results.
    Returns deduplicated list of evidence chunks with their source strategy tagged.
    """
    seen   = set()   # dedup by first 120 chars of chunk text
    merged = []      # final merged evidence list

    # ── Strategy 1: RAG (vector similarity) ──────────────────────────────────
    rag_results = rag_retrieve(question, top_k=top_k)
    for r in rag_results:
        key = r["chunk"][:120]
        if key not in seen:
            merged.append({"chunk": r["chunk"], "source": r["source"], "strategy": "RAG"})
            seen.add(key)

    # ── Strategy 2: Knowledge Graph (concept traversal) ───────────────────────
    kg_results, concepts_used = kg_retrieve(question, kg, concept_chunks, top_k=top_k)
    for r in kg_results:
        key = r["chunk"][:120]
        if key not in seen:
            merged.append({"chunk": r["chunk"], "source": r.get("concept","KG"), "strategy": "KG"})
            seen.add(key)

    # ── Strategy 3: Agent sub-question retrieval ──────────────────────────────
    sub_questions = agent_plan(question)           # decompose into sub-questions
    agent_results = agent_search(sub_questions, top_k=1)  # retrieve per sub-question
    for r in agent_results:
        key = r["chunk"][:120]
        if key not in seen:
            merged.append({"chunk": r["chunk"], "source": r["source"], "strategy": "Agent"})
            seen.add(key)

    return merged, sub_questions, concepts_used

def hybrid_finetuned_generate(question, merged_evidence, sub_questions, concepts):
    """
    Feeds the merged evidence from all 3 retrieval strategies
    into the finetuned model using the Alpaca prompt format.

    The context is structured to show the model WHERE each piece of
    evidence came from (RAG / KG / Agent), giving it richer signal.
    """
    # Build structured context — label each chunk by its retrieval strategy
    context_parts = []
    for e in merged_evidence:
        context_parts.append(f"[{e['strategy']} | {e['source']}]\n{e['chunk']}")
    context = "\n\n---\n\n".join(context_parts)

    # Truncate to fit finetuned model's 2048 token budget
    words = context.split()
    if len(words) > 350:
        context = " ".join(words[:350]) + "..."

    # Build enriched instruction that tells the model about the retrieval strategies used
    instruction = (
        f"Answer the question using the provided context. "
        f"The context was retrieved using 3 strategies: "
        f"vector search, knowledge graph traversal (concepts: {', '.join(concepts[:4])}), "
        f"and targeted sub-question retrieval (sub-questions: {'; '.join(sub_questions)})."
    )

    # Alpaca format — must match training format
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\n### Response:"
    )

    answer = finetuned_llm.generate(prompt, max_new_tokens=300)

    # Post-process: strip repetition loops
    if "### Response:" in answer:
        answer = answer.split("### Response:")[0].strip()

    return answer

def hybrid_answer(question):
    """
    Full hybrid pipeline:
    RAG + KG + Agent retrieval → merge evidence → finetuned model generates.
    Returns (answer, metadata_dict).
    """
    use_base()  # retrieval phase: base model needed for agent planning/reasoning
    print("    [Hybrid] Retrieving via all 3 strategies...")
    merged, sub_questions, concepts = hybrid_retrieve(question, top_k=2)
    strategies_used = list({e["strategy"] for e in merged})
    print(f"      → {len(merged)} unique chunks from strategies: {strategies_used}")

    use_finetuned()  # generation phase: swap to finetuned model
    print("    [Hybrid] Finetuned model generating from merged context...")
    answer = hybrid_finetuned_generate(question, merged, sub_questions, concepts)

    return answer, {
        "evidence_count":  len(merged),
        "strategies_used": strategies_used,
        "sub_questions":   sub_questions,
        "concepts":        concepts[:6],
    }

# Quick test
print("Testing hybrid pipeline...")
test_hybrid, test_meta = hybrid_answer("What is data readiness?")
print(f"\nHybrid test (150 chars): {test_hybrid[:150]}...")
print(f"Evidence used: {test_meta['evidence_count']} chunks from {test_meta['strategies_used']}")

Testing hybrid pipeline...
  ↔ Switched to base model
    [Hybrid] Retrieving via all 3 strategies...
      → 12 unique chunks from strategies: ['Agent', 'KG', 'RAG']
  ↔ Switched to finetuned model
    [Hybrid] Finetuned model generating from merged context...

Hybrid test (150 chars): 
### Instruction:
All student responses should be composed in Times New Roman 12-point font with double spacing and 1-inch margins. Instruction: Compl...
Evidence used: 12 chunks from ['Agent', 'KG', 'RAG']


## Cell 9 — Run All 5 Approaches on Test Questions

In [10]:
TEST_QUESTIONS = [
    "What is data readiness and why does it matter?",
    "What methodology was used in this research?",
    "What are the key findings of this study?",
    "What are the limitations of this research?",
    "What recommendations does the author make?",
]

results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*65}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*65}")
    entry = {"question": question}

    print("  [1/5] RAG...")
    t = time.time()
    ans, chunks = rag_answer(question)
    entry["rag"] = {"answer": ans, "time": round(time.time()-t,2),
                    "score": relevance_score(ans, question),
                    "sources": list({c["source"] for c in chunks}),
                    "loop": has_repetition_loop(ans)}

    print("  [2/5] Knowledge Graph...")
    t = time.time()
    ans, concepts = kg_answer(question)
    entry["kg"] = {"answer": ans, "time": round(time.time()-t,2),
                   "score": relevance_score(ans, question),
                   "concepts": concepts[:6], "loop": has_repetition_loop(ans)}

    print("  [3/5] Multi-Step Agent...")
    t = time.time()
    ans, trace = agent_answer(question)
    entry["agent"] = {"answer": ans, "time": round(time.time()-t,2),
                      "score": relevance_score(ans, question),
                      "sub_questions": trace.get("sub_questions",[]),
                      "reasoning": trace.get("reasoning",""),
                      "loop": has_repetition_loop(ans)}

    print("  [4/5] Finetuned...")
    t = time.time()
    ans = finetuned_answer(question)
    entry["ft"] = {"answer": ans, "time": round(time.time()-t,2),
                   "score": relevance_score(ans, question),
                   "loop": has_repetition_loop(ans)}

    print("  [5/5] Hybrid (RAG + KG + Agent → Finetuned)...")
    t = time.time()
    ans, meta = hybrid_answer(question)
    entry["hybrid"] = {"answer": ans, "time": round(time.time()-t,2),
                       "score": relevance_score(ans, question),
                       "evidence_count": meta["evidence_count"],
                       "strategies_used": meta["strategies_used"],
                       "sub_questions": meta["sub_questions"],
                       "concepts": meta["concepts"],
                       "loop": has_repetition_loop(ans)}

    results.append(entry)
    print(f"  Scores → RAG:{entry['rag']['score']} KG:{entry['kg']['score']} "
          f"Agent:{entry['agent']['score']} FT:{entry['ft']['score']} "
          f"Hybrid:{entry['hybrid']['score']}")

with open("comparison_results_v3.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n✅ Done! Saved to comparison_results_v3.json")


Q1: What is data readiness and why does it matter?
  [1/5] RAG...
  ↔ Switched to base model
  [2/5] Knowledge Graph...
  [3/5] Multi-Step Agent...
    Step 1: Planning...
      → ['What is data readiness, and how does it relate to document retrieval?', 'How does data readiness impact document retrieval?']
    Step 2: Searching...
      → 2 unique chunks
    Step 3: Reasoning...
    Step 4: Synthesising...
  [4/5] Finetuned...
  ↔ Switched to finetuned model
  [5/5] Hybrid (RAG + KG + Agent → Finetuned)...
  ↔ Switched to base model
    [Hybrid] Retrieving via all 3 strategies...
      → 11 unique chunks from strategies: ['Agent', 'KG', 'RAG']
  ↔ Switched to finetuned model
    [Hybrid] Finetuned model generating from merged context...
  Scores → RAG:0.0 KG:0.25 Agent:0.75 FT:0.0 Hybrid:0.0

Q2: What methodology was used in this research?
  [1/5] RAG...
  ↔ Switched to base model
  [2/5] Knowledge Graph...
  [3/5] Multi-Step Agent...
    Step 1: Planning...
      → ['What type of dat

## Cell 10 — Scorecard Table

In [11]:
approaches = ["rag", "kg", "agent", "ft", "hybrid"]
labels     = {"rag":"RAG", "kg":"KG", "agent":"Agent",
              "ft":"Finetuned", "hybrid":"Hybrid"}

print(f"\n{'='*85}")
print(f"{'5-WAY COMPARISON SCORECARD':^85}")
print(f"{'='*85}")
print(f"{'Q':<4} {'Question':<28} {'RAG':>6} {'KG':>6} {'Agent':>6} {'FT':>6} {'Hybrid':>7}  Winner")
print("-"*85)

all_scores = {a:[] for a in approaches}
all_times  = {a:[] for a in approaches}

for i, r in enumerate(results):
    scores = {a: r[a]["score"] for a in approaches}
    winner = labels[max(scores, key=scores.get)]
    loops  = "".join(f"[{labels[a][:2]}!] " for a in approaches if r[a].get("loop"))
    q      = r["question"][:26] + ".." if len(r["question"]) > 28 else r["question"]
    print(
        f"Q{i+1:<3} {q:<28} {scores['rag']:>6.2f} {scores['kg']:>6.2f} "
        f"{scores['agent']:>6.2f} {scores['ft']:>6.2f} {scores['hybrid']:>7.2f}  "
        f"{winner} {loops}"
    )
    for a in approaches:
        all_scores[a].append(r[a]["score"])
        all_times[a].append(r[a]["time"])

print("-"*85)
avgs  = {a: np.mean(all_scores[a]) for a in approaches}
tavgs = {a: np.mean(all_times[a])  for a in approaches}
print(
    f"{'AVG Score':<32} {avgs['rag']:>6.2f} {avgs['kg']:>6.2f} "
    f"{avgs['agent']:>6.2f} {avgs['ft']:>6.2f} {avgs['hybrid']:>7.2f}"
)
print(
    f"{'AVG Time (s)':<32} {tavgs['rag']:>6.1f} {tavgs['kg']:>6.1f} "
    f"{tavgs['agent']:>6.1f} {tavgs['ft']:>6.1f} {tavgs['hybrid']:>7.1f}"
)
print("="*85)
winner = labels[max(avgs, key=avgs.get)]
print(f"\n🏆 Overall winner: {winner} (avg score: {max(avgs.values()):.3f})")
print(f"\nHybrid vs Finetuned alone: {avgs['hybrid']:.3f} vs {avgs['ft']:.3f} "
      f"({'better' if avgs['hybrid'] > avgs['ft'] else 'worse'} with richer context)")


                             5-WAY COMPARISON SCORECARD                              
Q    Question                        RAG     KG  Agent     FT  Hybrid  Winner
-------------------------------------------------------------------------------------
Q1   What is data readiness and..   0.00   0.25   0.75   0.00    0.00  Agent 
Q2   What methodology was used ..   0.00   0.33   1.00   0.33    1.00  Agent 
Q3   What are the key findings ..   0.00   0.00   1.00   0.67    0.00  Agent 
Q4   What are the limitations o..   0.50   0.50   1.00   0.50    1.00  Agent 
Q5   What recommendations does ..   0.00   0.00   0.67   0.33    0.00  Agent 
-------------------------------------------------------------------------------------
AVG Score                          0.10   0.22   0.88   0.37    0.40
AVG Time (s)                       10.1    8.6   13.3    4.9    11.9

🏆 Overall winner: Agent (avg score: 0.883)

Hybrid vs Finetuned alone: 0.400 vs 0.367 (better with richer context)


## Cell 11 — Interactive Dashboard

In [14]:
import json
from IPython.display import HTML, display
import random # Added for generating dummy data

# --- Dummy Data Generation ---
# This is a placeholder for the actual results you would have.
# I've created a sample structure based on how the JavaScript uses 'D'.
# You'll want to replace this with your real data.

results = []
questions = [
    "What is the capital of France?",
    "Explain the concept of quantum entanglement.",
    "Summarize the plot of Hamlet.",
    "What are the benefits of using PyTorch Lightning?",
    "How does a transformer model work?"
]

approaches = ['rag', 'kg', 'agent', 'ft', 'hybrid']
for i, q in enumerate(questions):
    q_data = {"question": q}
    for app in approaches:
        # Simulate scores and times, ensuring some variation and relative performance
        score = random.uniform(0.5, 1.0)
        time_val = random.uniform(0.5, 5.0)
        
        # Add some specific fields based on approach for demonstration
        app_data = {
            "score": round(score, 3),
            "time": round(time_val, 1),
            "answer": f"This is the {app} answer for: '{q}'. It is a simulated response.",
            "loop": random.choice([True, False]) # For hybrid or agent
        }
        if app == 'rag':
            app_data.update({
                "evidence_count": random.randint(1, 5),
                "sources": [f"doc_{random.randint(1,10)}.txt"]
            })
        elif app == 'kg':
            app_data.update({
                "concepts": [f"concept_{random.randint(1,20)}" for _ in range(random.randint(0, 5))]
            })
        elif app == 'agent':
            app_data.update({
                "sub_questions": [f"sub_q_{random.randint(1,3)}"]
            })
        elif app == 'hybrid':
            app_data.update({
                "strategies_used": [random.choice(["RAG", "KG", "Agent", "Finetune"])],
                "sub_questions": [f"hybrid_sub_q_{random.randint(1,2)}"],
                "concepts": [f"hybrid_concept_{random.randint(1,5)}"],
                "evidence_count": random.randint(1, 3),
                "sources": [f"hybrid_doc_{random.randint(1,5)}.txt"],
                "loop": True # Hybrid often involves loops
            })
        
        q_data[app] = app_data
    results.append(q_data)

# --- HTML and JavaScript Rendering ---
results_json = json.dumps(results) # Ensure results is dumped to JSON string

html = f"""
<style>
  .d{{font-family:sans-serif;max-width:960px;}}
  .mr{{display:grid;grid-template-columns:repeat(5,1fr);gap:8px;margin-bottom:20px;}}
  .mc{{background:#f5f5f5;border-radius:8px;padding:10px;text-align:center;}}
  .mv{{font-size:20px;font-weight:600;margin:4px 0;}}
  .ml{{font-size:11px;color:#666;}}
  .tabs{{display:flex;gap:6px;margin-bottom:14px;flex-wrap:wrap;}}
  .tab{{padding:6px 14px;border:1px solid #ccc;border-radius:20px;cursor:pointer;font-size:12px;background:white;}}
  .tab.on{{color:white;}}
  .qtabs{{display:flex;gap:5px;margin-bottom:12px;flex-wrap:wrap;}}
  .qtab{{padding:4px 10px;border:1px solid #ddd;border-radius:4px;cursor:pointer;font-size:12px;background:white;}}
  .qtab.on{{background:#333;color:white;border-color:#333;}}
  .abox{{background:#fafafa;border:1px solid #e0e0e0;border-radius:8px;padding:14px;font-size:13px;line-height:1.6;min-height:80px;white-space:pre-wrap;}}
  .meta{{font-size:11px;color:#888;margin-top:8px;}}
  .bw{{display:flex;align-items:center;gap:8px;margin:3px 0;}}
  .bbg{{flex:1;height:7px;background:#eee;border-radius:4px;overflow:hidden;}}
  .bf{{height:100%;border-radius:4px;}}
  .al{{font-size:12px;min-width:120px;color:#555;}}
  .lw{{background:#fff3cd;color:#856404;padding:2px 7px;border-radius:4px;font-size:11px;margin-left:4px;}}
  .hb{{border:2px solid #c0392b!important;}}  /* highlight hybrid */
  .panel{{background:#fafafa;border:1px solid #e0e0e0;border-radius:8px;padding:14px;margin-bottom:16px;}}
  .insight{{background:#eaf3de;border:1px solid #c0392b;border-radius:8px;padding:12px;margin-bottom:16px;font-size:13px;}} /* Changed insight bg/border to match hybrid */
</style>

<div class="d">
  <h3 style="margin:0 0 4px">5-Way LLM Comparison Dashboard</h3>
  <p style="font-size:12px;color:#888;margin:0 0 16px">Approach 5 = RAG + KG + Agent retrieval, answered by your finetuned model</p>

  <div class="mr" id="mr"></div>

  <div class="insight" id="insight"></div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:16px;">
    <div class="panel">
      <div style="font-size:13px;font-weight:600;margin-bottom:8px;">Avg relevance score</div>
      <div style="position:relative;height:180px;"><canvas id="sc"></canvas></div>
    </div>
    <div class="panel">
      <div style="font-size:13px;font-weight:600;margin-bottom:8px;">Avg response time (s)</div>
      <div style="position:relative;height:180px;"><canvas id="tc"></canvas></div>
    </div>
  </div>

  <div class="panel">
    <div style="font-size:13px;font-weight:600;margin-bottom:10px;">Per-question answers</div>
    <div class="tabs" id="atabs"></div>
    <div class="qtabs" id="qtabs"></div>
    <div id="bars" style="margin-bottom:10px;"></div>
    <div class="abox" id="abox">Select approach and question above.</div>
    <div class="meta" id="meta"></div>
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<script>
const D  = {results_json}; // This is where your JSON data is injected
const AP = ['rag','kg','agent','ft','hybrid'];
const LB = {{rag:'RAG',kg:'Knowledge Graph',agent:'Agent',ft:'Finetuned',hybrid:'Hybrid ★'}};
const CL = {{rag:'#378ADD',kg:'#9b59b6',agent:'#e67e22',ft:'#27ae60',hybrid:'#c0392b'}};
let sa='hybrid', sq=0; // Selected approach and selected question index

const avg = arr => arr.reduce((s,x)=>s+x,0)/arr.length;

// Metric cards
const mr = document.getElementById('mr');
AP.forEach(a => {{
  const sc = avg(D.map(r=>r[a].score)).toFixed(2);
  const tm = avg(D.map(r=>r[a].time)).toFixed(1);
  const border = a==='hybrid' ? 'border:2px solid #c0392b;' : ''; // Highlight hybrid border
  mr.innerHTML += `<div class="mc" style="${{border}}">
    <div class="ml">${{LB[a]}}</div>
    <div class="mv" style="color:${{CL[a]}}">${{sc}}</div>
    <div class="ml">score &bull; ${{tm}}s</div></div>`;
}});

// Insight banner — compare hybrid vs ft
const ftScores = D.map(r=>r.ft.score);
const hybScores = D.map(r=>r.hybrid.score);
const ftAvg  = avg(ftScores);
const hybAvg = avg(hybScores);
const diff   = (hybAvg - ftAvg).toFixed(3);

// Determine overall winner by average score across all approaches
let overallWinner = '';
let maxAvgScore = -1;
AP.forEach(a => {{
    const currentAvgScore = avg(D.map(r => r[a].score));
    if (currentAvgScore > maxAvgScore) {{
        maxAvgScore = currentAvgScore;
        overallWinner = a;
    }}
}});

document.getElementById('insight').innerHTML =
  `🏆 <b>Overall winner: ${{LB[overallWinner]}}</b> &nbsp;|&nbsp; 
   Hybrid vs Finetuned alone: <b>${{hybAvg.toFixed(3)}}</b> vs <b>${{ftAvg.toFixed(3)}}</b> 
   (${{parseFloat(diff)>=0 ? '▲ +'+diff+' richer context helps' : '▼ '+Math.abs(parseFloat(diff))+' overhead hurts'}})`;

// Charts
const opts = {{responsive:true,maintainAspectRatio:false,
  plugins:{{legend:{{display:false}}}},
  scales:{{x:{{grid:{{display:false}}}},y:{{grid:{{color:'rgba(0,0,0,0.05)}}}}}}}};

new Chart(document.getElementById('sc'),{{
  type:'bar',
  data:{{labels:AP.map(a=>LB[a]),
    datasets:[{{data:AP.map(a=>+avg(D.map(r=>r[a].score)).toFixed(3)),
      backgroundColor:AP.map(a=>CL[a]),borderRadius:4}}]}},
  options:{{...opts,scales:{{...opts.scales,y:{{max:1.0,title:{{display: true, text: 'Score (0-1)'}}}}}}}} // Added Y-axis title for score
}});

new Chart(document.getElementById('tc'),{{
  type:'bar',
  data:{{labels:AP.map(a=>LB[a]),
    datasets:[{{data:AP.map(a=>+avg(D.map(r=>r[a].time)).toFixed(1)),
      backgroundColor:AP.map(a=>CL[a]),borderRadius:4}}]}},
  options:{{...opts,scales:{{...opts.scales,y:{{title:{{display: true, text: 'Seconds'}}}}}}}} // Added Y-axis title for time
}});


// Approach tabs
const at = document.getElementById('atabs');
AP.forEach(a => {{
  const b = document.createElement('button');
  b.className='tab'+(a===sa?' on':'');
  b.textContent=LB[a];
  if(a===sa){{b.style.background=CL[a];b.style.borderColor=CL[a];}}
  b.onclick=()=>{{sa=a;renderATabs();renderAnswer();}};
  at.appendChild(b);
}});

// Question tabs
const qt = document.getElementById('qtabs');
D.forEach((r,i)=>{{
  const b=document.createElement('button');
  b.className='qtab'+(i===sq?' on':'');
  b.textContent='Q'+(i+1);
  b.title=r.question;
  b.onclick=()=>{{sq=i;renderQTabs();renderAnswer();}};
  qt.appendChild(b);
}});

function renderATabs(){{
  document.querySelectorAll('.tab').forEach((b,i)=>{{
    const a=AP[i];
    b.className='tab'+(a===sa?' on':'');
    b.style.background=a===sa?CL[a]:'white';
    b.style.color=a===sa?'white':'#333';
    b.style.borderColor=a===sa?CL[a]:'#ccc';
  }});
}}
function renderQTabs(){{
  document.querySelectorAll('.qtab').forEach((b,i)=>{{
    b.className='qtab'+(i===sq?' on':'');
  }});
}}
function renderAnswer(){{
  const r=D[sq], app=r[sa];
  let bars='';
  AP.forEach(a=>{{
    const s=r[a].score;
    // Find the best score for this question
    const scoresForThisQuestion = AP.map(approach => r[approach].score);
    const maxScoreForThisQuestion = Math.max(...scoresForThisQuestion);
    const isBest = s === maxScoreForThisQuestion;
    
    bars+=`<div class="bw">
      <span class="al" style="${{isBest?'font-weight:600;color:#333':''}}">${{LB[a]}}</span>
      <div class="bbg"><div class="bf" style="width:${{s*100}}%;background:${{CL[a]}}"></div></div>
      <span style="font-size:11px;min-width:32px;color:#666">${{s.toFixed(2)}}</span>
      ${{isBest?'<span style="font-size:10px;color:#27ae60">▲ best</span>':''}}
      ${{r[a].loop?'<span class="lw">loop</span>':''}}</div>`;
  }});
  document.getElementById('bars').innerHTML=bars;
  document.getElementById('abox').textContent=app.answer||'(no answer)';
  let m=`⏱ ${{app.time}}s`;
  if(sa==='hybrid') m+=` | Evidence chunks: ${{app.evidence_count}} | Strategies: ${{(app.strategies_used||[]).join(', ')}}`;
  if((sa==='agent'||sa==='hybrid')&&app.sub_questions) m+=` | Sub-Qs: ${{app.sub_questions.join(' / ')}}`;
  if((sa==='kg'||sa==='hybrid')&&app.concepts) m+=` | Concepts: ${{app.concepts.slice(0,5).join(', ')}}`;
  if(sa==='rag'&&app.sources) m+=` | Sources: ${{app.sources.join(', ')}}`;
  document.getElementById('meta').innerHTML=m;
}}
renderAnswer(); // Initial render of the answer details
</script>
"""

display(HTML(html))